In [58]:
!pip install arxiv

In [59]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [60]:
!pip install  wikipedia
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [61]:
wiki.name

'wikipedia'

In [62]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

In [63]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OpenAIEmbeddings

loader = WebBaseLoader('https://docs.langchain.com/langsmith/home')
docs = loader.load()
documents= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,OpenAIEmbeddings())
retriever = vectordb.as_retriever()
retriever


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11b1e0d60>, search_kwargs={})

In [64]:
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",   # underscore, no spaces, correct spelling
    "Search for information about Langsmith. For any questions about Langsmith, you must use this tool"
)

In [65]:
retriever_tool.name

'langsmith_search'

In [66]:
##arxiv
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [67]:
tools = [wiki,arxiv,retriever_tool]

In [68]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model ="gpt-3.5-turbo-0125",temperature=0)

In [69]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [70]:
import langchain
import langchain_core
import langchain_community
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)

langchain: 0.3.25
langchain_core: 0.3.84
langchain_community: 0.3.25


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor


agent = create_tool_calling_agent(llm, tools, prompt)



In [74]:
from langchain.agents import AgentExecutor
agent_executor= AgentExecutor(agent=agent,tools=tools,verboe=True)
agent_executor

AgentExecutor(verbose=False, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanM

In [75]:
agent_executor.invoke({"input":"Tell me about Langsmith"})

{'input': 'Tell me about Langsmith',
 'output': 'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM (Large Language Model) applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place with their agent stack. LangSmith offers various tools such as Fleet for designing and deploying AI agents visually without writing code, Prompt engineering for iterating on prompts with built-in versioning and collaboration, LangSmith CLI for querying and managing traces, datasets, and experiments from the terminal, and Studio for using a visual interface to design, test, and refine applications end-to-end.\n\nThe platform setup of LangSmith can be done in a managed cloud, self-hosted environment, or hybrid to match infrastructure and compliance needs. It meets high standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. LangSmith combines observability, evaluation, d